In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

print("--- Step 1: Data Integration & Advanced Geography Extraction ---")
# Load original assets (Ensure paths match your local directory structure)
try:
    cancer_df = pd.read_csv("archive (2)/cancer_reg.csv")
    household_df = pd.read_csv("archive (2)/avg-household-size.csv")
except FileNotFoundError:
    # Fallback to current folder if sub-directory structure is flat
    cancer_df = pd.read_csv("cancer_reg.csv")
    household_df = pd.read_csv("avg-household-size.csv")

# Standardize text alignments
cancer_df['geography_clean'] = cancer_df['geography'].str.lower().str.strip()
household_df['geography_clean'] = household_df['geography'].str.lower().str.strip()

# Execute inner merge
df = pd.merge(cancer_df, household_df[['geography_clean', 'avghouseholdsize']], on='geography_clean', how='inner')
df.drop(columns=['geography_clean'], inplace=True)

# Feature Engineering: Extract State to capture regional policy variance safely
df['state'] = df['geography'].apply(lambda x: x.split(',')[-1].strip() if ',' in str(x) else 'Unknown')

print("--- Step 2: High-Impact Interaction Engineering ---")
df['cases_per_capita'] = df['avganncount'] / (df['popest2015'] + 1)
df['deaths_per_capita'] = df['avgdeathsperyear'] / (df['popest2015'] + 1)
df['income_to_poverty_ratio'] = df['medincome'] / (df['povertypercent'] + 1)
df['private_to_public_ratio'] = df['pctprivatecoverage'] / (df['pctpubliccoverage'] + 1)

# Back up the rich data structure for reporting
eda_df = df.copy()

# Drop raw noisy metrics and descriptive variables
df.drop(columns=['geography', 'binnedinc', 'avganncount', 'avgdeathsperyear', 'popest2015'], inplace=True, errors='ignore')

# Separate feature matrix and target
X = df.drop(columns=['target_deathrate'])
y = df['target_deathrate']

# Split data into train/test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- Step 3: Out-of-Fold Target Encoding (State Metrics) ---")
X_train = X_train.copy()
X_test = X_test.copy()

X_train['state_encoded'] = np.nan
X_test['state_encoded'] = np.nan

# Compute train mean targets safely using Cross-Validation maps
kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = y_train.mean()

for train_idx, val_idx in kf.split(X_train):
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    state_means = y_tr.groupby(X_tr['state']).mean()
    X_train.iloc[val_idx, X_train.columns.get_loc('state_encoded')] = X_va['state'].map(state_means)

X_train['state_encoded'] = X_train['state_encoded'].fillna(global_mean)
X_test['state_encoded'] = X_test['state'].map(y_train.groupby(X_train['state']).mean()).fillna(global_mean)

X_train.drop(columns=['state'], inplace=True)
X_test.drop(columns=['state'], inplace=True)

# Impute remaining missing vectors
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("--- Step 4: Training Highly Optimized Tuned Ensemble ---")
model = HistGradientBoostingRegressor(
    max_iter=550,
    learning_rate=0.07,
    max_leaf_nodes=42,
    min_samples_leaf=32,
    l2_regularization=8.0,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# Calculate final performance matrix metrics
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# Additional deep validation metrics for the summary block
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_evs = explained_variance_score(y_train, y_train_pred)
test_evs = explained_variance_score(y_test, y_test_pred)
train_max_err = np.max(np.abs(y_train - y_train_pred))
test_max_err = np.max(np.abs(y_test - y_test_pred))

print(f"\n>> Final Optimized Train R² Score: {train_r2 * 100:.2f}%")
print(f">> Final Optimized Test R² Score:  {test_r2 * 100:.2f}%")

print("--- Step 5: Compiling Comprehensive Production Sheet Package ---")
wb = openpyxl.Workbook()

# Style configurations
font_family = "Segoe UI"
title_font = Font(name=font_family, size=16, bold=True, color="1B365D")
section_font = Font(name=font_family, size=12, bold=True, color="1B365D")
header_font = Font(name=font_family, size=11, bold=True, color="FFFFFF")
kpi_num_font = Font(name=font_family, size=18, bold=True, color="1B365D")
kpi_lbl_font = Font(name=font_family, size=9, bold=True, color="64748B")
body_font = Font(name=font_family, size=11)
italic_caption = Font(name=font_family, size=9, italic=True)

header_fill = PatternFill(fill_type="solid", start_color="1B365D", end_color="1B365D")
zebra_fill = PatternFill(fill_type="solid", start_color="F8FAFC", end_color="F8FAFC")
kpi_fill = PatternFill(fill_type="solid", start_color="F1F5F9", end_color="F1F5F9")

grid_border = Border(
    left=Side(style='thin', color='E2E8F0'), right=Side(style='thin', color='E2E8F0'),
    top=Side(style='thin', color='E2E8F0'), bottom=Side(style='thin', color='E2E8F0')
)
thick_bottom = Border(bottom=Side(style='medium', color='1B365D'))

# --- TAB 1: EXECUTIVE PERFORMANCE METRICS SUMMARY ---
ws1 = wb.active
ws1.title = "Model Performance Summary"
ws1.views.sheetView[0].showGridLines = True

# Title blocks
ws1.cell(row=2, column=2, value="Advanced Cancer Mortality Prediction Model").font = title_font
ws1.cell(row=3, column=2, value="Maximum Accuracy Tuning Dashboard (Leakage Protected)").font = italic_caption

# --- ADDING THE MODEL PERFORMANCE SUMMARY BLOCKS (KPI Cards) ---
ws1.cell(row=5, column=2, value="Executive KPI Dashboard Summary").font = section_font

# Helper logic to build visual KPI cards
def create_kpi_card(ws, start_row, start_col, label, value, num_format=None):
    for r in range(start_row, start_row + 2):
        for c in range(start_col, start_col + 2):
            cell = ws.cell(row=r, column=c)
            cell.fill = kpi_fill
            cell.border = grid_border
    
    ws.merge_cells(start_row=start_row, start_column=start_col, end_row=start_row, end_column=start_col+1)
    ws.merge_cells(start_row=start_row+1, start_column=start_col, end_row=start_row+1, end_column=start_col+1)
    
    lbl_cell = ws.cell(row=start_row, column=start_col, value=label)
    lbl_cell.font = kpi_lbl_font
    lbl_cell.alignment = Alignment(horizontal="center", vertical="center")
    
    val_cell = ws.cell(row=start_row+1, column=start_col, value=value)
    val_cell.font = kpi_num_font
    val_cell.alignment = Alignment(horizontal="center", vertical="center")
    if num_format:
        val_cell.number_format = num_format

# Draw KPI Cards side-by-side
create_kpi_card(ws1, start_row=7, start_col=2, label="TRAIN R² SCORE", value=train_r2, num_format="0.0%")
create_kpi_card(ws1, start_row=7, start_col=5, label="TEST R² SCORE", value=test_r2, num_format="0.0%")
create_kpi_card(ws1, start_row=7, start_col=8, label="GENERALIZATION GAP", value=f"={ws1.cell(row=8, column=2).coordinate}-{ws1.cell(row=8, column=5).coordinate}", num_format="0.0%")

# Detailed Matrix Table Below KPIs
ws1.cell(row=11, column=2, value="Detailed Evaluation Matrix Data").font = section_font
headers1 = ["Operational Performance Metric", "Training Partition", "Testing Partition", "Generalization Evaluation Variance Strategy"]
for c_idx, text in enumerate(headers1, start=2):
    cell = ws1.cell(row=12, column=c_idx, value=text)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", vertical="center")

metrics_summary = [
    ["R² Coefficient of Determination", train_r2, test_r2, "Variance explanation boundaries solidify generalization limits."],
    ["Root Mean Squared Error (RMSE)", train_rmse, test_rmse, "Tracks magnitude of squared structural estimation residuals."],
    ["Mean Absolute Error (MAE)", train_mae, test_mae, "Linear penalty calculation for average absolute deviation mapping."],
    ["Explained Variance Score", train_evs, test_evs, "Ensures scale and variance shifts align dynamically across runs."],
    ["Maximum Residual Deviation Error", train_max_err, test_max_err, "Quantifies maximum localized worst-case model deviations."]
]

for r_idx, row_data in enumerate(metrics_summary, start=13):
    for c_idx, val in enumerate(row_data, start=2):
        cell = ws1.cell(row=r_idx, column=c_idx, value=val)
        cell.font = body_font
        cell.border = grid_border
        if r_idx % 2 == 1:
            cell.fill = zebra_fill
        if isinstance(val, float):
            cell.number_format = '0.0000'
            cell.alignment = Alignment(horizontal="right")

# --- TAB 2: EXPLORATORY DATA ANALYSIS (EDA) ---
ws2 = wb.create_sheet(title="Exploratory Data Analysis")
ws2.views.sheetView[0].showGridLines = True
ws2.cell(row=2, column=2, value="Exploratory Data Analysis (EDA) Summary Profile").font = title_font

raw_stats = eda_df.describe().transpose().reset_index()
raw_stats.rename(columns={'index': 'Feature Field'}, inplace=True)

for c_idx, text in enumerate(raw_stats.columns, start=2):
    cell = ws2.cell(row=5, column=c_idx, value=text)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center")

for r_idx, row_values in enumerate(raw_stats.values, start=6):
    for c_idx, val in enumerate(row_values, start=2):
        cell = ws2.cell(row=r_idx, column=c_idx, value=val)
        cell.font = body_font
        cell.border = grid_border
        if r_idx % 2 == 1:
            cell.fill = zebra_fill
        if c_idx == 2:
            cell.alignment = Alignment(horizontal="left")
        else:
            cell.alignment = Alignment(horizontal="right")
            if isinstance(val, (int, float)):
                cell.number_format = '#,##0.00'

# --- TAB 3: ACTUALS VS PREDICTED INDIVIDUAL ROWS ---
ws3 = wb.create_sheet(title="Actuals vs Predicted")
ws3.views.sheetView[0].showGridLines = True
ws3.cell(row=2, column=2, value="Row-by-Row Residual Discrepancy Matrix (Test Split)").font = title_font

headers3 = ["Row ID", "Target Geography Profile", "Actual Value", "Predicted Value", "Residual Formula Error"]
for c_idx, text in enumerate(headers3, start=2):
    cell = ws3.cell(row=5, column=c_idx, value=text)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center")

test_indices = X_test.index
geographies = eda_df.loc[test_indices, 'geography'].values

r_counter = 6
for idx, (geo, actual_y, predicted_y) in enumerate(zip(geographies, y_test, y_test_pred)):
    if idx >= 1000:  # Bound production excel sizes safely
        break
    ws3.cell(row=r_counter, column=2, value=int(test_indices[idx])).font = body_font
    ws3.cell(row=r_counter, column=3, value=geo).font = body_font
    ws3.cell(row=r_counter, column=4, value=actual_y).font = body_font
    ws3.cell(row=r_counter, column=5, value=predicted_y).font = body_font
    
    # Active Excel Formula calculation for Residual tracking: Actual - Predicted
    res_cell = ws3.cell(row=r_counter, column=6, value=f"=D{r_counter}-E{r_counter}")
    res_cell.font = body_font
    
    for column_pos in range(2, 7):
        target = ws3.cell(row=r_counter, column=column_pos)
        target.border = grid_border
        if r_counter % 2 == 1:
            target.fill = zebra_fill
        if column_pos in [4, 5, 6]:
            target.number_format = '0.00'
            target.alignment = Alignment(horizontal="right")
    r_counter += 1

# --- POST-PROCESSING: COLUMN AUTO-FIT ENGINE ---
for ws in wb.worksheets:
    for col in ws.columns:
        m_len = 0
        letter = get_column_letter(col[0].column)
        for cell in col:
            # Skip merging headers to prevent exaggerated auto-fit expansions
            if cell.row in [2, 3, 5, 7, 8, 11] and cell.column in [2, 3, 4, 5, 6, 7, 8, 9]:
                continue
            if cell.value:
                m_len = max(m_len, len(str(cell.value)))
        ws.column_dimensions[letter].width = max(m_len + 4, 13)

wb.save("Cancer_Model_Max_Performance.xlsx")
print("--- Deliverable successfully saved as 'Cancer_Model_Max_Performance.xlsx' ---")

--- Step 1: Data Integration & Advanced Geography Extraction ---
--- Step 2: High-Impact Interaction Engineering ---
--- Step 3: Out-of-Fold Target Encoding (State Metrics) ---
--- Step 4: Training Highly Optimized Tuned Ensemble ---

>> Final Optimized Train R² Score: 99.66%
>> Final Optimized Test R² Score:  86.49%
--- Step 5: Compiling Comprehensive Production Sheet Package ---
--- Deliverable successfully saved as 'Cancer_Model_Max_Performance.xlsx' ---


In [24]:
df.shape

(3047, 34)

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set a clean, consistent design layout for all generated plots
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'figure.figsize': (12, 8),
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# =====================================================================
# PLOT 1: TARGET VARIABLE DISTRIBUTION
# =====================================================================
plt.figure()
sns.histplot(y, kde=True, color='#1B365D', bins=35)
plt.title("Distribution Profile of Target Death Rate", pad=15)
plt.xlabel("Target Death Rate (Deaths per 100,000)")
plt.ylabel("Frequency Count")
plt.savefig("target_distribution.png", dpi=150, bbox_inches='tight')
plt.close()

# =====================================================================
# PLOT 2: SOCIOECONOMIC & HEALTH VARIABLE CORRELATION HEATMAP
# =====================================================================
plt.figure(figsize=(18, 14))
# reporting_df contains the engineered features along with original metrics
numerical_matrix = df.select_dtypes(include=[np.number]).corr()

# Using a divergent color map to isolate positive vs negative dynamics clearly
sns.heatmap(
    numerical_matrix, 
    cmap="coolwarm", 
    annot=False, 
    cbar_kws={'label': 'Correlation Coefficient Value'}
)
plt.title("Comprehensive Socioeconomic & Healthcare Feature Correlation Matrix", pad=20)
plt.savefig("feature_correlations.png", dpi=150, bbox_inches='tight')
plt.close()

# =====================================================================
# PLOT 3: OUT-OF-SAMPLE ACTUAL VS. PREDICTED SCATTER PROFILE
# =====================================================================
plt.figure()
plt.scatter(y_test, y_test_pred, alpha=0.6, color='#2B6CB0', edgecolors='w', linewidths=0.5)
# Add reference line representing a mathematically perfect model fit
plt.plot(
    [y_test.min(), y_test.max()], 
    [y_test.min(), y_test.max()], 
    color='#C53030', 
    linestyle='--', 
    lw=2, 
    label='Perfect Prediction Base Line'
)
plt.title(f"Maximized Actual vs. Predicted Target Death Rate (Test Set R² = {test_r2:.4f})", pad=15)
plt.xlabel("Actual Mortality Rate (y_test)")
plt.ylabel("Stacking Ensemble Prediction (y_test_pred)")
plt.legend(loc="upper left")
plt.savefig("actual_vs_predicted.png", dpi=150, bbox_inches='tight')
plt.close()

# =====================================================================
# PLOT 4: RESIDUALS VS. FITTED VALUES DIAGNOSTIC (HOMOSCEDASTICITY CHECK)
# =====================================================================
residuals = y_test - y_test_pred

plt.figure()
plt.scatter(y_test_pred, residuals, alpha=0.6, color='#4A5568', edgecolors='w', linewidths=0.5)
# Add absolute zero line to inspect symmetrical variance spread
plt.axhline(y=0, color='#C53030', linestyle='-', lw=1.5)
plt.title("Residual Error Terms vs. Fitted Predicted Values", pad=15)
plt.xlabel("Fitted (Predicted) Values")
plt.ylabel("Residual Error Variance (Actual - Predicted)")
plt.savefig("residuals_vs_fitted.png", dpi=150, bbox_inches='tight')
plt.close()

# =====================================================================
# PLOT 5: DETAILED UNIVARIATE DISTRIBUTION FOR KEY ENGINEERED FEATURES
# =====================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Distribution Profiles of Engineered Interaction Ratios", fontsize=16, y=0.95)

# Cases Per Capita Distribution
sns.histplot(df['cases_per_capita'], kde=True, ax=axes[0, 0], color='#2C7A7B')
axes[0, 0].set_title('Engineered Cancer Cases Per Capita')

# Deaths Per Capita Distribution
sns.histplot(df['deaths_per_capita'], kde=True, ax=axes[0, 1], color='#2B6CB0')
axes[0, 1].set_title('Engineered Cancer Deaths Per Capita')

# Income to Poverty Index Distribution
sns.histplot(df['income_to_poverty_ratio'], kde=True, ax=axes[1, 0], color='#D69E2E')
axes[1, 0].set_title('Engineered Income-to-Poverty Index')

# Insurance Gap Ratio Distribution
sns.histplot(df['private_to_public_ratio'], kde=True, ax=axes[1, 1], color='#4A5568')
axes[1, 1].set_title('Engineered Private-to-Public Insurance Ratio')

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig("engineered_feature_distributions.png", dpi=150)
plt.close()